# LatentDriver Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_colab_runner.ipynb)

Use this notebook as a single Colab terminal launcher. It keeps experiment logic in repo-owned CLI scripts and writes debug bundles to Drive so failures can be pulled locally with `rclone`.


## Operating Model

1. Run the bootstrap cell to clone or fast-forward `main`.
2. Mount Drive and bind the persistent artifact layout.
3. Pick one `PROFILE`.
4. Run the profile cell.

Default profile is `full-eval-dry-run` because full preprocessing is assumed to be complete. Use `full-preprocess` only if you intentionally want to resume or rebuild preprocessing.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        lines: list[str] = []
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, text=True, capture_output=True, check=False)
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {printable}")
    return completed.stdout


if not REPO_DIR.exists():
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"REPO_DIR exists but is not a git checkout: {REPO_DIR}")
else:
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

head = run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], stream=False).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "branch": REPO_BRANCH, "head": head}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE_ROOT = "/content/drive/MyDrive/waymax_research"
WAYMO_GCS_ROOT = "gs://waymo_open_dataset_motion_v_1_1_0"

sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_BASE_ROOT)
os.environ["LATENTDRIVER_WAYMO_DATASET_ROOT"] = WAYMO_GCS_ROOT
os.environ["LATENTDRIVER_DEBUG_ROOT"] = str(Path(binding["drive_root"]) / "debug_runs")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

print(json.dumps({
    "binding": binding,
    "LATENTDRIVER_WAYMO_DATASET_ROOT": os.environ["LATENTDRIVER_WAYMO_DATASET_ROOT"],
    "LATENTDRIVER_DEBUG_ROOT": os.environ["LATENTDRIVER_DEBUG_ROOT"],
}, indent=2, sort_keys=True))


In [ ]:
# Available profiles. This is safe and does not run heavy setup.
run([sys.executable, "scripts/colab_canary.py", "--list-profiles"], cwd=REPO_DIR)


In [ ]:
# Change only this cell for normal use.
PROFILE = "full-eval-dry-run"
MODEL = "latentdriver_t2_j3"
SEED = 0
VIS = "video"

# Leave this True: the CLI installs runtime only for profiles that need it.
AUTO_INSTALL_RUNTIME = True

# Use these only when needed.
FORCE_INSTALL_RUNTIME = False
DOWNLOAD_CHECKPOINTS = False
DRY_RUN_CANARY = False  # True means write a debug bundle with planned commands but do not execute them.

print(json.dumps({
    "PROFILE": PROFILE,
    "MODEL": MODEL,
    "SEED": SEED,
    "VIS": VIS,
    "AUTO_INSTALL_RUNTIME": AUTO_INSTALL_RUNTIME,
    "FORCE_INSTALL_RUNTIME": FORCE_INSTALL_RUNTIME,
    "DOWNLOAD_CHECKPOINTS": DOWNLOAD_CHECKPOINTS,
    "DRY_RUN_CANARY": DRY_RUN_CANARY,
}, indent=2, sort_keys=True))


In [ ]:
cmd = [
    sys.executable,
    "scripts/colab_canary.py",
    "--profile",
    PROFILE,
    "--model",
    MODEL,
    "--seed",
    str(SEED),
    "--vis",
    VIS,
]
if AUTO_INSTALL_RUNTIME:
    cmd.append("--auto-install-runtime")
if FORCE_INSTALL_RUNTIME:
    cmd.append("--install-runtime")
if DOWNLOAD_CHECKPOINTS:
    cmd.append("--download-checkpoints")
if DRY_RUN_CANARY:
    cmd.append("--dry-run")

run(cmd, cwd=REPO_DIR)


In [ ]:
# Optional: inspect stable debug pointers after a run.
from pathlib import Path
import json

DEBUG_ROOT = Path(os.environ["LATENTDRIVER_DEBUG_ROOT"])
for pointer_name in ["LATEST.json", "LATEST_FAILURE.json"]:
    pointer_path = DEBUG_ROOT / pointer_name
    if pointer_path.exists():
        print(f"{pointer_name}: {pointer_path}")
        print(json.dumps(json.loads(pointer_path.read_text(encoding="utf-8")), indent=2, sort_keys=True))
    else:
        print(f"{pointer_name}: missing")

print(f"Stable latest alias: {DEBUG_ROOT / 'latest'}")
print(f"Stable latest failure alias: {DEBUG_ROOT / 'latest_failure'}")
